# Log dataset suitability

Everything is configured and run from this notebook: where the logs are, how a line is
read, how templates are mined, which thresholds apply, and whether the result is kept.

```
Stage 1  normalize   raw logs      ->  13-column CSV
Stage 2  templates   CSV           ->  per-file D1, D2, D3, D5 and the near-duplicate ratio
Stage 3  entities    CSV           ->  D4, cross-component structure
         scores      stage output  ->  one score vector
         verdicts    score vector  ->  a verdict per task, and the figure
```

Edit **section 1** and run the notebook top to bottom. With no logs present it writes a
small synthetic dataset, so it runs end to end out of the box.

For a dataset too large to sit in a notebook, the same pipeline runs from a shell:
`python src/run_pipeline.py --config configs/mylogs.yml`.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import paths
import registry
from normalize import normalize, report
from templates import DrainConfig, compute_global_neardup, mine_templates
from entities import measure_entities
from scores import extract_scores, measure_separability
from verdicts import APPLICATION_CLASSES, THRESHOLDS, build_verdict_rows

pd.set_option("display.width", 160)
print(f"package: {ROOT}")
print(f"corpus:  {registry.CORPUS_FILE.name}, "
      f"{len(registry.DATASETS)} dataset(s) recorded so far")

## 1. Configuration

Everything the run depends on. Nothing below this cell needs editing.

**`STAGE1`** describes the log format, and is the only dataset-specific part.
`host` decides the nodes of the cross-component graph that D4 measures, and `labels`
decides whether D3 can be computed at all. Every key is documented at the top of
`src/normalize.py`; the options are

| Key | Values |
|---|---|
| `host.from` | `path_part` (with `index`), `filename`, `line_regex` (with `pattern`), `fixed` (with `value`) |
| `timestamp.from` | `auto`, `regex` (with `pattern` and `strptime`), `none` |
| `labels.from` | `none`, `line_numbers`, `id_regex`, `time_windows`, `file_glob` |

**`DRAIN`** controls template mining. Template counts are not comparable across
different settings, so keep them fixed when comparing datasets.

**`TAU`** are the thresholds the verdicts gate on. They were calibrated in the study;
changing one invalidates that calibration, but the effect is visible immediately.

In [ ]:
# What to profile ---------------------------------------------------------------
SLUG = "mylogs"                       # names the output directory and the corpus entry
DATA_DIR = paths.DATA / SLUG          # where the raw logs are

# Stage 1: how to read this log format ------------------------------------------
STAGE1 = {
    "dataset": SLUG,
    "input_dir": str(DATA_DIR),
    "pattern": "**/*.log",            # glob for log files under input_dir
    "exclude": [],                    # globs to skip, for example ["**/metrics/**"]
    "host":      {"from": "path_part", "index": 0},
    "timestamp": {"from": "auto"},
    "labels":    {"from": "line_numbers", "path": str(DATA_DIR / "labels.json")},
}

# Stage 2: template mining ------------------------------------------------------
DRAIN = DrainConfig(
    depth=4,                          # parse tree depth
    sim_th=0.5,                       # similarity needed to join a cluster
    max_children=100,                 # children per internal node
    max_clusters=500,                 # cap per source file; 0 disables it
    parametrize_numeric_tokens=True,  # replace numbers before matching
)
WRITE_LINE_ASSIGNMENTS = True         # drain_lines.csv; needed for D3, large on big data
SKIP_TELEMETRY_FILES = True           # skip per-second metric dumps
SAMPLE_FRACTION = 1.0                 # fraction of source files to process

# Verdicts ----------------------------------------------------------------------
TAU = {"D1": 0.40, "D2": 0.15, "D3": 0.25, "D4": 0.40}

# Keep the result in the corpus file when the run finishes ----------------------
SAVE_TO_CORPUS = False

STAGE2_DIR = paths.dataset_dir(SLUG) / "stage2"
STAGE3_DIR = paths.stage3_dir(SLUG)
NORMALIZED = paths.normalized_csv(SLUG)
print(json.dumps(STAGE1, indent=2))
print(f"\ndrain: {DRAIN}")
print(f"thresholds: {TAU}")

### A dataset to work on

With no logs at `DATA_DIR`, this writes a small synthetic one: four files over three
hosts, with public and private addresses, a shared request id, and a few failed-login
lines marked as attacks. Delete the cell once you point the configuration at real logs.

In [ ]:
if not DATA_DIR.is_dir():
    import random
    import uuid

    rng = random.Random(7)
    request_id = str(uuid.UUID(int=rng.getrandbits(128)))
    public_ips = ["203.0.113.10", "198.51.100.7"]
    attack_lines = {}
    for host, name in (("web", "auth.log"), ("web", "app.log"),
                       ("db", "auth.log"), ("gateway", "syslog.log")):
        lines, attacks = [], []
        for i in range(220):
            stamp = f"2024-03-{1 + i % 3:02d} {i % 24:02d}:{i % 60:02d}:{(i * 7) % 60:02d}"
            draw = rng.random()
            if draw < 0.35:
                lines.append(f"{stamp} session opened for user u{rng.randint(1, 9)}")
            elif draw < 0.60:
                ip = rng.choice(public_ips + ["10.0.0.1"])
                lines.append(f"{stamp} connection from {ip} port {rng.randint(1024, 65535)} accepted")
            elif draw < 0.75:
                lines.append(f"{stamp} request {request_id} handled by worker {rng.randint(1, 4)}")
            elif draw < 0.90:
                lines.append(f"{stamp} job {rng.randint(1, 4)} finished in {rng.randint(1, 900)} ms")
            else:
                ip = rng.choice(public_ips)
                lines.append(f"{stamp} failed password for invalid user admin from {ip} ssh2")
                attacks.append(len(lines))
        path = DATA_DIR / host / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        attack_lines[f"{host}/{name}"] = attacks
    (DATA_DIR / "labels.json").write_text(
        json.dumps({"attack_line_numbers": attack_lines}), encoding="utf-8")
    print(f"wrote a synthetic dataset to {DATA_DIR}")
else:
    print(f"using the logs already at {DATA_DIR}")

## 2. Stage 1: normalize

Reads the raw files and writes one row per log line in the 13-column schema. This is the
only place a log format is interpreted. Read the summary: no attack lines means D3 cannot
be computed, and a single host means D4 is undefined.

In [ ]:
summary = normalize(STAGE1)
report(summary)

## 3. Stage 2: templates and the per-file metrics

One Drain3 miner per source file. Every line gets a template id, and the id sequence is
the shared input for D1, D2, D3 and D5. The dataset-wide near-duplicate ratio follows,
because it needs the templates of every file at once and it completes D1.

In [ ]:
stage2 = mine_templates(
    normalized_csv=NORMALIZED,
    output_dir=STAGE2_DIR,
    config=DRAIN,
    write_line_assignments=WRITE_LINE_ASSIGNMENTS,
    skip_metrics_files=SKIP_TELEMETRY_FILES,
    sample_fraction=SAMPLE_FRACTION,
)
print(f"source files: {stage2['total_source_files']}")
print(f"templates:    {stage2['total_templates']}")
print(f"lines:        {stage2['total_lines_processed']:,}")

neardup = compute_global_neardup(STAGE2_DIR / "drain_templates.csv")
(STAGE2_DIR / "d1_global_neardup.json").write_text(json.dumps(neardup, indent=2),
                                                   encoding="utf-8")
print(f"near-duplicate ratio: {neardup['global_neardup_ratio']:.4f} "
      f"over {neardup['total_distinct_templates']} distinct templates")

## 4. Stage 3: cross-component structure

Reads the raw message text rather than the templates, because templating removes exactly
the identifiers this stage needs. Two channels are reported and never combined: address
sharing between hosts, and sharing of a participation identifier.

In [ ]:
stage3 = measure_entities(normalized_csv=NORMALIZED, output_dir=STAGE3_DIR)
print(f"hosts:      {stage3['host_count']}")
print(f"host pairs: {stage3['cross_host_pair_count']}")
pd.DataFrame(stage3["entity_types"]).T[["total_distinct", "hosts_with_entities",
                                        "pairs_sharing", "sharing_ratio"]]

## 5. The score vector

Label separability is measured on the global distribution of (template, label) over the
whole dataset, which makes it independent of how the logs are split into files. The
per-file metrics are then aggregated to the dataset by median, and the near-duplicate
correction is applied to D1.

In [ ]:
measure_separability([SLUG], paths.VERDICTS / "d3_information.json")
measured = extract_scores(None, SLUG, "stage2")

pd.DataFrame([{k: measured.get(k) for k in
               ("D1", "D1_robust", "D2", "D3", "D3_U", "D3_purity",
                "D4_conn", "D4_part", "D5", "n_files", "n_attack_files")}])

## 6. Verdicts

The measured dataset joins the corpus in memory, together with any dataset recorded
earlier. `TAU` from section 1 is applied here: change a threshold there and rerun this
cell to see which verdicts move.

In [ ]:
registry.DATASETS.setdefault(SLUG, SLUG)
registry.CORPUS_SCORES[SLUG] = {
    "D1": measured.get("D1"), "D1_robust": measured.get("D1_robust"),
    "D2": measured.get("D2"), "D3": measured.get("D3"),
    "D4": measured.get("D4_conn"), "D4_part": measured.get("D4_part"),
    "D5": measured.get("D5"),
}
if measured.get("D3_U") is not None:
    registry.D3_INFO[SLUG] = {"U": measured["D3_U"], "purity": measured["D3_purity"]}

rows = build_verdict_rows(TAU)

DIMS = ["D1", "D2", "D3", "D4", "D5"]
scores = {}
for row in rows:
    entry = scores.setdefault(row["dataset"], {d: None for d in DIMS})
    for d in DIMS:
        if entry[d] is None:
            entry[d] = row[f"{d}_score"]

order = [spec["display_name"] for spec in APPLICATION_CLASSES.values()]
verdict_table = pd.DataFrame(rows).pivot(index="dataset", columns="application",
                                         values="verdict")[order]

SHADE = {
    "SUITABLE": "background-color: #E6F4EA; color: #1E6B2E",
    "UNSUITABLE": "background-color: #FCE8E6; color: #A4231C",
    "INDETERMINATE": "background-color: #F1F1F1; color: #6B6B6B",
}
verdict_table.style.map(lambda v: SHADE.get(v, ""))

In [ ]:
print(f"{SLUG}:")
for task in order:
    print(f"  {task:<28}{verdict_table.loc[SLUG, task]}")

print()
counts = verdict_table.stack().value_counts()
total = int(counts.sum())
for verdict in ("SUITABLE", "UNSUITABLE", "INDETERMINATE"):
    print(f"{verdict:<15}{int(counts.get(verdict, 0)):>4}  of {total} (dataset, task) pairs")

### Which threshold decided a verdict

For every dataset that is not SUITABLE, the necessary dimensions that failed or could not
be measured, with the distance to the threshold. A small margin means the verdict would
move under a slightly different threshold.

In [ ]:
reasons = []
for row in rows:
    if row["verdict"] == "SUITABLE":
        continue
    for dim in row["necessary_dims"].split(","):
        value, tau = row[f"{dim}_score"], row[f"{dim}_thresh"]
        if value is None:
            reasons.append({"dataset": row["dataset"], "task": row["application"],
                            "dimension": dim, "score": None, "threshold": tau,
                            "margin": None, "reason": "not measurable"})
        elif value < tau:
            reasons.append({"dataset": row["dataset"], "task": row["application"],
                            "dimension": dim, "score": round(value, 3), "threshold": tau,
                            "margin": round(value - tau, 3), "reason": "below threshold"})

if reasons:
    reasons = pd.DataFrame(reasons).sort_values(["margin", "dataset"],
                                                na_position="last").reset_index(drop=True)
    print(f"{len(reasons)} failing or unmeasurable necessary dimensions")
    display(reasons.head(20))
else:
    print("Every dataset is SUITABLE for every task, so no threshold decided against one.")

## 7. The verdict figure

Each panel is one task, plotting the **necessary dimensions** of that task against their
thresholds. The shaded corner is where every necessary dimension is met, so a point inside
it is SUITABLE and a point outside failed on whichever threshold it sits below. Marker
colour repeats the verdict, and the dataset measured above is ringed. With more datasets
recorded, each becomes a point here, so the figure shows where a dataset sits relative to
the others.

The panels follow the necessary sets in `src/verdicts.py`, so changing a necessary set or
a threshold in section 1 changes the figure with it.

In [ ]:
VERDICT_COLOR = {"SUITABLE": "#2E7D32", "UNSUITABLE": "#C62828", "INDETERMINATE": "#8A8A8A"}
DIM_LABEL = {
    "D1": "D1  structural diversity",
    "D2": "D2  event-sequence diversity",
    "D3": "D3  label separability",
    "D4": "D4  cross-component linkability",
    "D5": "D5  semantic richness",
}
LO, HI = -0.10, 1.16          # axis limits, shared by every panel
PAD = 0.03                    # keep labels this far inside the axes


def place_labels(ax, points, names, ylo=LO, yhi=HI):
    """Write each dataset name next to its marker, inside the axes.

    Candidate offsets are tried in order (right, then progressively further
    above and below) and the first that neither collides with an already
    placed label nor leaves the axes is used.
    """
    placed = []
    steps = [0.0] + [s * d for s in (0.042, 0.084, 0.126, 0.168, 0.21) for d in (1, -1)]
    for x, y, name in sorted(zip([p[0] for p in points], [p[1] for p in points], names),
                            key=lambda t: (-t[1], t[0])):
        for dy_off in steps:
            y_text = y + dy_off
            if not (ylo + PAD <= y_text <= yhi - PAD):
                continue
            if any(abs(y_text - py) < 0.036 and abs(x - px) < 0.30 for px, py in placed):
                continue
            break
        else:
            y_text = min(max(y, ylo + PAD), yhi - PAD)
        ax.annotate(name, (x, y), (x + 0.016, y_text), fontsize=7.5, color="#333333",
                    va="center", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=0.4, color="#BBBBBB",
                                    shrinkA=0, shrinkB=2))
        placed.append((x, y_text))


def place_labels_1d(ax, xs, names, y=0.5, ylo=0.0, yhi=1.0):
    """Fan names above and below a single line of points."""
    order = sorted(zip(xs, names))
    rungs = max(1, (len(order) + 1) // 2)
    # Keep labels close to the line when there are few points, and spread them
    # only as far as the axes allow when there are many.
    span = min(0.07, (yhi - y - PAD) / rungs)
    for rank, (x, name) in enumerate(order):
        step = rank // 2 + 1
        up = rank % 2 == 0
        y_text = y + span * step * (1 if up else -1)
        ax.annotate(name, (x, y), (x, y_text), fontsize=7.5, color="#333333",
                    ha="center", va="bottom" if up else "top", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=0.4, color="#BBBBBB",
                                    shrinkA=0, shrinkB=3))


def panel(ax, spec):
    necessary = spec["necessary"]
    display = spec["display_name"]
    dx = necessary[0]
    dy = necessary[1] if len(necessary) > 1 else None
    tx = TAU.get(dx, THRESHOLDS[dx])
    one_dim = dy is None
    ylo, yhi = (0.0, 1.0) if one_dim else (LO, HI)

    # The region in which every necessary dimension is met.
    if one_dim:
        ax.axvspan(tx, HI, facecolor="#2E7D32", alpha=0.07, zorder=0)
    else:
        ty = TAU.get(dy, THRESHOLDS[dy])
        ax.add_patch(plt.Rectangle((tx, ty), HI - tx, HI - ty, facecolor="#2E7D32",
                                   alpha=0.07, edgecolor="#2E7D32", lw=0.8, ls=":",
                                   zorder=0))
        ax.axhline(ty, color="#555555", ls=":", lw=1.0, zorder=1)
        ax.text(HI - 0.012, ty + 0.012, f"tau = {ty:.2f}", fontsize=7.5, color="#555555",
                ha="right", va="bottom", zorder=2)
    ax.axvline(tx, color="#555555", ls=":", lw=1.0, zorder=1)
    ax.text(tx + 0.012, yhi - 0.012, f"tau = {tx:.2f}", fontsize=7.5, color="#555555",
            ha="left", va="top", rotation=90, zorder=2)

    points, names, unmeasured = [], [], []
    for dataset, values in scores.items():
        x = values[dx]
        y = values[dy] if dy else 0.5
        if x is None or (dy and y is None):
            unmeasured.append(dataset)
            continue
        focus = dataset == SLUG
        ax.scatter(x, y, s=150 if focus else 70,
                   facecolor=VERDICT_COLOR[verdict_table.loc[dataset, display]],
                   alpha=0.9 if focus else 0.85,
                   edgecolor="#111111" if focus else "white",
                   linewidth=1.6 if focus else 1.1, zorder=4 if focus else 3)
        points.append((x, y))
        names.append(dataset)

    if one_dim:
        place_labels_1d(ax, [p[0] for p in points], names, ylo=ylo, yhi=yhi)
    else:
        place_labels(ax, points, names)

    ax.set_title(f"{display}\nnecessary: {' and '.join(necessary)}", fontsize=11)
    ax.set_xlabel(DIM_LABEL[dx], fontsize=9)
    ax.set_xlim(LO, HI)
    ax.set_ylim(ylo, yhi)
    ax.grid(alpha=0.22, lw=0.5)
    ax.tick_params(labelsize=8)
    if one_dim:
        ax.set_yticks([])
    else:
        ax.set_ylabel(DIM_LABEL[dy], fontsize=9)
    if unmeasured:
        ax.text(0.99, 0.015, "not measurable: " + ", ".join(sorted(unmeasured)),
                transform=ax.transAxes, fontsize=7.5, color="#8A8A8A",
                ha="right", va="bottom", zorder=4,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))


tasks = list(APPLICATION_CLASSES.values())
ncols = 2
nrows = (len(tasks) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13.5, 5.8 * nrows), layout="constrained")
flat = axes.ravel()
for ax, spec in zip(flat, tasks):
    panel(ax, spec)
for ax in flat[len(tasks):]:
    ax.axis("off")

handles = [plt.Line2D([], [], marker="o", ls="", markersize=9, markerfacecolor=colour,
                      markeredgecolor="white", label=verdict)
           for verdict, colour in VERDICT_COLOR.items()]
handles.append(plt.Line2D([], [], marker="o", ls="", markersize=10,
                          markerfacecolor="none", markeredgecolor="#111111",
                          markeredgewidth=1.6, label=f"{SLUG} (measured here)"))
fig.legend(handles=handles, loc="outside lower center", ncol=4, frameon=False, fontsize=10)
fig.suptitle("Necessary dimensions per task", fontsize=13)
plt.show()

In [ ]:
target = paths.FIGURES / f"verdicts_{SLUG}.png"
target.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(target, dpi=200, bbox_inches="tight")
print(f"written: {target}")

## 8. Keep the result

Sections 2 to 7 wrote this dataset's own stage output, one entry in the shared
`outputs/verdicts/d3_information.json`, and the figure; the corpus file is untouched.
Setting `SAVE_TO_CORPUS = True` in section 1 writes the score vector into the corpus
file, so the dataset is included on every later run and by `python src/verdicts.py`. The corpus starts
empty and fills up as datasets are profiled, which is what makes the figure comparative
once there is more than one.

In [ ]:
if SAVE_TO_CORPUS:
    written = registry.save_dataset(
        name=SLUG,
        slug=SLUG,
        scores=registry.CORPUS_SCORES[SLUG],
        d3_information=registry.D3_INFO.get(SLUG),
        extras={"global_neardup": neardup.get("global_neardup_ratio")},
    )
    print(f"{SLUG} written to {written}")
else:
    print("SAVE_TO_CORPUS is False, the corpus file is unchanged.")
    print("The dataset is in the corpus for this session only.")

## What the scores mean

| Dimension | Low value means |
|---|---|
| D1 | a few templates dominate the vocabulary: boilerplate, near-duplicate templates |
| D2 | event order is nearly deterministic, so sequence models have little to learn |
| D3 | attack lines reuse the templates of normal lines, so templates cannot separate them |
| D4 | hosts rarely share identifiers, so cross-host reconstruction has few links to follow |

`docs/METHOD.md` defines each dimension, the aggregation and where each threshold came
from. Per-file values are in `outputs/<slug>/stage2/d1_metrics.csv`, and the host pairs
behind D4 in `outputs/<slug>/stage3_d4/d4_cross_host.csv`.